<a href="https://colab.research.google.com/github/ADRIANVM117/data-science-portfolio/blob/main/scenario-portfolio-lab/notebooks/06_portfolio_construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

El botón abre esta notebook en Google Colab. Allí ejecuta primero la siguiente celda de preparación para obtener el código y los reportes públicos; no descarga datos de mercado ni solicita credenciales.

# 06. Construcción de una cartera candidata

## 1. ¿Qué le pedimos al optimizador?

En 05, el modelo exponencial quedó como referencia de trabajo a una sesión. Ahora usaremos esos escenarios para comparar dos asignaciones en una sola fecha de corte: pesos iguales y mínimo CVaR.

El mandato de esta primera versión es el siguiente:

- Ambas carteras usan el mismo universo elegible de 125 instrumentos en USD y los mismos escenarios exponenciales a una sesión, con semivida de 1,102 sesiones.
- Ambas están plenamente invertidas, sin posiciones cortas ni apalancamiento. Fijaremos el tope por instrumento antes de optimizar.
- Para cada escenario, el retorno de cartera es la suma de los retornos simples de los instrumentos multiplicados por sus pesos. La pérdida se expresa como el negativo de ese retorno.
- Usamos CVaR al 95% como criterio inicial: la pérdida media en el peor 5% de la masa de probabilidad. Con probabilidades desiguales, ese 5% no equivale necesariamente al 5% de las filas.
- Reportaremos el retorno medio estimado y la concentración para entender qué cambia al minimizar el CVaR. No se impondrá una meta de retorno en esta primera versión.

La notebook 06 termina con una tabla de pesos y una comparación de riesgo dentro de los escenarios. La evaluación histórica con rebalanceo y costos queda para 07.

**Siguiente pregunta:** ¿Cómo convertimos los retornos conjuntos de cada escenario en una distribución de pérdidas de cartera?

## 2. ¿Cuánto gana o pierde la cartera en cada escenario?

Los **pesos de cartera** asignan el capital entre instrumentos: un peso por cada una de las 125 columnas. Las **probabilidades de escenario** asignan masa a las fechas históricas: un peso por cada una de las 2,204 filas. Son objetos distintos: que una fecha reciente tenga más probabilidad no cambia directamente la asignación entre instrumentos, y que una cartera tenga un peso alto en un instrumento no hace más probable ningún escenario.

Para un escenario histórico $t$, el retorno simple de cartera es $r_{p,t}=r_t^\top w$, donde $r_t$ es el vector de retornos simples conjuntos y $w$ el vector de pesos de cartera. La pérdida se define como $L_t=-r_{p,t}$. En este bloque usamos probabilidades exponenciales sobre las filas históricas con semivida de 1,102 sesiones y una cartera plenamente invertida de pesos iguales; todavía no calculamos VaR, CVaR ni optimizamos.

In [1]:
from pathlib import Path
import site
import numpy as np

def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        for root_candidate in (candidate, candidate / 'scenario-portfolio-lab', candidate / 'data-science-portfolio' / 'scenario-portfolio-lab'):
            if (root_candidate / 'config' / 'universe.csv').is_file() and (root_candidate / 'src' / 'scenario_portfolio_lab').is_dir():
                return root_candidate
    raise RuntimeError('Run this notebook inside the complete scenario-portfolio-lab repository.')

ROOT = find_project_root()
project_site_packages = ROOT / '.venv' / 'Lib' / 'site-packages'
if project_site_packages.is_dir():
    site.addsitedir(str(project_site_packages))
import polars as pl

CUT_RUN = ROOT / 'data' / 'published' / 'cut_2026-10-09_run_2026-10-11_full_window_final'
returns_path = CUT_RUN / 'gold' / 'eligible_simple_total_return_matrix.parquet'
eligible_path = CUT_RUN / 'reports' / 'eligible_universe.csv'
returns_with_initial_null = pl.read_parquet(returns_path)
eligible_catalog = pl.read_csv(eligible_path).filter(pl.col('universe_status') == 'ELIGIBLE')
# The first row has no approved preceding price. Remove that known initial-return row explicitly, not with a generic null filter.
initial_return_row = returns_with_initial_null.head(1)
if returns_with_initial_null.height != 2205 or initial_return_row.select(returns_with_initial_null.columns[1:]).null_count().to_numpy().sum() != 125:
    raise AssertionError('Expected exactly one initial all-null return row in the published matrix.')
returns_frame = returns_with_initial_null.slice(1, 2204)

scenario_dates = returns_frame['session_date'].to_list()
scenario_symbols = returns_frame.columns[1:]
simple_return_scenarios = returns_frame.select(scenario_symbols).to_numpy().astype(float)
SCENARIO_HALF_LIFE = 1102
scenario_count, instrument_count = simple_return_scenarios.shape
scenario_age = scenario_count - 1 - np.arange(scenario_count)
scenario_weights_exponential = np.exp(-np.log(2.0) * scenario_age / SCENARIO_HALF_LIFE)
scenario_weights_exponential /= scenario_weights_exponential.sum()
portfolio_weights_equal = np.full(instrument_count, 1.0 / instrument_count)

# Matrix multiplication: rows are historical joint scenarios; columns are instruments.
portfolio_returns_equal = simple_return_scenarios @ portfolio_weights_equal
portfolio_losses_equal = -portfolio_returns_equal

scenario_index_check = 0
explicit_contributions = simple_return_scenarios[scenario_index_check] * portfolio_weights_equal
block2_controls = {
    'published_cut_and_dimensions': scenario_count == 2204 and instrument_count == 125 and scenario_dates[-1].isoformat() == '2026-10-09',
    'dates_unique_and_increasing': scenario_dates == sorted(scenario_dates) and len(set(scenario_dates)) == scenario_count,
    'symbols_match_eligible_catalog_in_order': scenario_symbols == eligible_catalog['symbol'].to_list() and len(set(scenario_symbols)) == instrument_count,
    'initial_null_row_removed_and_remaining_returns_finite': returns_frame.null_count().to_numpy().sum() == 0 and np.isfinite(simple_return_scenarios).all() and np.all(simple_return_scenarios > -1.0),
    'scenario_probabilities_normalized': np.isfinite(scenario_weights_exponential).all() and np.all(scenario_weights_exponential > 0.0) and np.isclose(scenario_weights_exponential.sum(), 1.0, atol=1e-12, rtol=1e-12),
    'portfolio_weights_fully_invested': np.isfinite(portfolio_weights_equal).all() and np.all(portfolio_weights_equal >= 0.0) and np.isclose(portfolio_weights_equal.sum(), 1.0, atol=1e-12, rtol=1e-12),
    'scenario_return_matches_explicit_contributions': np.isclose(portfolio_returns_equal[scenario_index_check], explicit_contributions.sum(), atol=1e-12, rtol=1e-12),
    'loss_is_negative_portfolio_return': np.allclose(portfolio_losses_equal, -portfolio_returns_equal, atol=1e-12, rtol=1e-12),
}
if not all(block2_controls.values()):
    raise AssertionError([name for name, passed in block2_controls.items() if not passed])

scenario_preview_indices = [0, 1, 2, scenario_count - 2, scenario_count - 1]
scenario_preview = pl.DataFrame({
    'scenario_date': [scenario_dates[i] for i in scenario_preview_indices],
    'scenario_probability': scenario_weights_exponential[scenario_preview_indices],
    'equal_weight_portfolio_return': portfolio_returns_equal[scenario_preview_indices],
    'equal_weight_portfolio_loss': portfolio_losses_equal[scenario_preview_indices],
})
display(scenario_preview)
display(pl.DataFrame({'control': list(block2_controls), 'result': ['PASS' if passed else 'FAIL' for passed in block2_controls.values()]}))
print(f'Escenarios: {scenario_count}; instrumentos: {instrument_count}; semivida: {SCENARIO_HALF_LIFE} sesiones. Cada fecha mostrada identifica la procedencia histórica de un escenario conjunto.')


scenario_date,scenario_probability,equal_weight_portfolio_return,equal_weight_portfolio_loss
date,f64,f64,f64
2018-01-03,0.00021,0.004566,-0.004566
2018-01-04,0.00021,0.004631,-0.004631
2018-01-05,0.00021,0.006075,-0.006075
2026-10-08,0.000838,-0.001203,0.001203
2026-10-09,0.000838,0.007032,-0.007032


control,result
str,str
"""published_cut_and_dimensions""","""PASS"""
"""dates_unique_and_increasing""","""PASS"""
"""symbols_match_eligible_catalog…","""PASS"""
"""initial_null_row_removed_and_r…","""PASS"""
"""scenario_probabilities_normali…","""PASS"""
"""portfolio_weights_fully_invest…","""PASS"""
"""scenario_return_matches_explic…","""PASS"""
"""loss_is_negative_portfolio_ret…","""PASS"""


Escenarios: 2204; instrumentos: 125; semivida: 1102 sesiones. Cada fecha mostrada identifica la procedencia histórica de un escenario conjunto.


## 3. ¿Cuánto pierde la cartera de referencia en la cola?

Supongamos que, al ordenar las pérdidas de mayor a menor, una fila acumula 4.8% de masa de probabilidad y la siguiente aporta 0.4%. Para formar exactamente la cola de 5%, incluimos toda la primera masa y sólo 0.2 puntos porcentuales de la segunda. La probabilidad original de esa segunda fecha no cambia: sólo su contribución al cálculo de cola es parcial.

El **VaR al 95%** es un umbral de pérdida: el cuantil 95% definido mediante la inversa de la CDF ponderada. El **CVaR al 95%** es la pérdida media ponderada en la peor masa de 5%. Por eso no promediamos simplemente todas las filas con pérdida mayor o igual al VaR: si varias filas empatan en el umbral, ese método puede incluir más de 5% de masa. La atribución parcial a fechas concretas empatadas puede variar con el orden, pero el CVaR resultante es el mismo.

In [2]:
TAIL_PROBABILITY = 0.05
loss_order_ascending = np.argsort(portfolio_losses_equal, kind='stable')
losses_ascending = portfolio_losses_equal[loss_order_ascending]
probabilities_ascending = scenario_weights_exponential[loss_order_ascending]
weighted_cdf_ascending = np.cumsum(probabilities_ascending)
var_index_ascending = int(np.searchsorted(weighted_cdf_ascending, 1.0 - TAIL_PROBABILITY, side='left'))
var_95_loss = float(losses_ascending[var_index_ascending])

loss_order_descending = np.argsort(-portfolio_losses_equal, kind='stable')
tail_losses = portfolio_losses_equal[loss_order_descending]
tail_probabilities_original = scenario_weights_exponential[loss_order_descending]
tail_probability_before = np.concatenate(([0.0], np.cumsum(tail_probabilities_original[:-1])))
tail_probability_included = np.clip(TAIL_PROBABILITY - tail_probability_before, 0.0, tail_probabilities_original)
cvar_95_loss = float((tail_probability_included * tail_losses).sum() / TAIL_PROBABILITY)

# Equivalent Rockafellar-Uryasev-style identity for this discrete weighted distribution.
cvar_95_independent = float(var_95_loss + (scenario_weights_exponential * np.maximum(portfolio_losses_equal - var_95_loss, 0.0)).sum() / TAIL_PROBABILITY)
weighted_mean_return = float(scenario_weights_exponential @ portfolio_returns_equal)
tail_boundary_index = int(np.flatnonzero(tail_probability_included > 0.0)[-1])
tail_window_indices = np.arange(max(0, tail_boundary_index - 2), min(scenario_count, tail_boundary_index + 3))
tail_window_original_indices = loss_order_descending[tail_window_indices]
tail_window = pl.DataFrame({
    'scenario_date': [scenario_dates[i] for i in tail_window_original_indices],
    'portfolio_loss': tail_losses[tail_window_indices],
    'original_scenario_probability': tail_probabilities_original[tail_window_indices],
    'tail_mass_included': tail_probability_included[tail_window_indices],
    'partially_included': (tail_probability_included[tail_window_indices] > 0.0) & (tail_probability_included[tail_window_indices] < tail_probabilities_original[tail_window_indices]),
})

block3_controls = {
    'tail_mass_is_exactly_five_percent': np.isclose(tail_probability_included.sum(), TAIL_PROBABILITY, atol=1e-12, rtol=1e-12),
    'included_mass_is_bounded_by_original_probability': np.all(tail_probability_included >= 0.0) and np.all(tail_probability_included <= tail_probabilities_original + 1e-15),
    'cvar_is_at_least_var': cvar_95_loss + 1e-12 >= var_95_loss,
    'cvar_matches_independent_identity': np.isclose(cvar_95_loss, cvar_95_independent, atol=1e-12, rtol=1e-12),
    'var_is_weighted_cdf_inverse': (weighted_cdf_ascending[var_index_ascending] >= 1.0 - TAIL_PROBABILITY - 1e-12) and (var_index_ascending == 0 or weighted_cdf_ascending[var_index_ascending - 1] < 1.0 - TAIL_PROBABILITY + 1e-12),
}
if not all(block3_controls.values()):
    raise AssertionError([name for name, passed in block3_controls.items() if not passed])

risk_summary = pl.DataFrame({
    'metric': ['weighted_mean_return', 'VaR_95_loss', 'CVaR_95_loss'],
    'value_decimal': [weighted_mean_return, var_95_loss, cvar_95_loss],
    'value_percent_of_capital': [100.0 * weighted_mean_return, 100.0 * var_95_loss, 100.0 * cvar_95_loss],
    'horizon': ['one session', 'one session', 'one session'],
})
display(risk_summary)
display(tail_window)
display(pl.DataFrame({'control': list(block3_controls), 'result': ['PASS' if passed else 'FAIL' for passed in block3_controls.values()]}))
print(f'La fila de frontera completa la masa de cola: índice ordenado={tail_boundary_index}, masa incluida={tail_probability_included[tail_boundary_index]:.8f}, probabilidad original={tail_probabilities_original[tail_boundary_index]:.8f}.')
print('Estas cifras describen la cola de la distribución construida: no son una pérdida máxima ni una garantía para la siguiente sesión.')


metric,value_decimal,value_percent_of_capital,horizon
str,f64,f64,str
"""weighted_mean_return""",0.00077,0.077043,"""one session"""
"""VaR_95_loss""",0.018375,1.837541,"""one session"""
"""CVaR_95_loss""",0.028356,2.835637,"""one session"""


scenario_date,portfolio_loss,original_scenario_probability,tail_mass_included,partially_included
date,f64,f64,f64,bool
2022-11-16,0.018631,0.000454,0.000454,false
2026-03-03,0.018455,0.000761,0.000761,false
2018-03-23,0.018375,0.000217,0.000124,true
2025-11-13,0.018369,0.000727,0.0,false
2023-10-18,0.018235,0.000524,0.0,false


control,result
str,str
"""tail_mass_is_exactly_five_perc…","""PASS"""
"""included_mass_is_bounded_by_or…","""PASS"""
"""cvar_is_at_least_var""","""PASS"""
"""cvar_matches_independent_ident…","""PASS"""
"""var_is_weighted_cdf_inverse""","""PASS"""


La fila de frontera completa la masa de cola: índice ordenado=124, masa incluida=0.00012405, probabilidad original=0.00021711.
Estas cifras describen la cola de la distribución construida: no son una pérdida máxima ni una garantía para la siguiente sesión.


## 4. ¿Cuál es la referencia para un universo de ocho ETFs?

Fijamos el universo del experimento en **SPY, VEA, EEM, TLT, BIL, GLD, USO y VNQ**. Según la revisión local, todos son ETFs elegibles en USD, directos y no apalancados. La disponibilidad concreta en IBKR sigue pendiente de verificar; esta definición no constituye una orden ni una recomendación.

| ETF | Función en el experimento | Solapamiento que debemos reconocer |
| --- | --- | --- |
| SPY | Acciones estadounidenses amplias | Incluye sectores US; por eso no añadimos XLE, XLK, XLF o XLV al núcleo. |
| VEA | Acciones desarrolladas fuera de EE. UU. | Puede compartir multinacionales con SPY, pero cubre un bloque geográfico distinto. |
| EEM | Acciones de mercados emergentes | Solapa parcialmente con una exposición asiática como AAXJ; elegimos sólo EEM. |
| TLT | Tesoro US de larga duración | Complementa, no sustituye, a BIL: ambos expresan duraciones muy distintas. |
| BIL | Letras del Tesoro US de 1–3 meses | Componente de muy corta duración; no representa una clase de activo nueva frente a TLT. |
| GLD | Oro | No añadimos IAU porque sería una exposición al oro muy cercana. |
| USO | Petróleo | Está relacionado económicamente con energía, pero no equivale a acciones energéticas como XLE. |
| VNQ | Bienes raíces cotizados de EE. UU. | Comparte mercado US con SPY; no implica diversificación garantizada. |

La referencia usa pesos iguales de 12.5%, sin posiciones cortas, margen ni apalancamiento. Sobre un presupuesto *paper* hipotético de USD 1 millón, cada monto objetivo es USD 125,000; son montos contables para interpretar pesos, no órdenes.

In [3]:
REFERENCE_EIGHT_SYMBOLS = ['SPY', 'VEA', 'EEM', 'TLT', 'BIL', 'GLD', 'USO', 'VNQ']
REFERENCE_BUDGET_USD = 1_000_000.0
reference_eight_indices = [scenario_symbols.index(symbol) for symbol in REFERENCE_EIGHT_SYMBOLS]
reference_eight_symbols = [scenario_symbols[index] for index in reference_eight_indices]
reference_eight_returns = simple_return_scenarios[:, reference_eight_indices]
reference_eight_weights = np.full(len(REFERENCE_EIGHT_SYMBOLS), 1.0 / len(REFERENCE_EIGHT_SYMBOLS))
reference_eight_target_amounts = REFERENCE_BUDGET_USD * reference_eight_weights

# Rows remain the same 2,204 joint scenarios; only the selected instrument columns and portfolio weights change.
reference_eight_portfolio_returns = reference_eight_returns @ reference_eight_weights
reference_eight_portfolio_losses = -reference_eight_portfolio_returns
reference_eight_order_ascending = np.argsort(reference_eight_portfolio_losses, kind='stable')
reference_eight_losses_ascending = reference_eight_portfolio_losses[reference_eight_order_ascending]
reference_eight_probabilities_ascending = scenario_weights_exponential[reference_eight_order_ascending]
reference_eight_var_index = int(np.searchsorted(np.cumsum(reference_eight_probabilities_ascending), 1.0 - TAIL_PROBABILITY, side='left'))
reference_eight_var_95_loss = float(reference_eight_losses_ascending[reference_eight_var_index])
reference_eight_order_descending = np.argsort(-reference_eight_portfolio_losses, kind='stable')
reference_eight_tail_losses = reference_eight_portfolio_losses[reference_eight_order_descending]
reference_eight_tail_probabilities = scenario_weights_exponential[reference_eight_order_descending]
reference_eight_tail_before = np.concatenate(([0.0], np.cumsum(reference_eight_tail_probabilities[:-1])))
reference_eight_tail_mass = np.clip(TAIL_PROBABILITY - reference_eight_tail_before, 0.0, reference_eight_tail_probabilities)
reference_eight_cvar_95_loss = float((reference_eight_tail_mass * reference_eight_tail_losses).sum() / TAIL_PROBABILITY)
reference_eight_cvar_independent = float(reference_eight_var_95_loss + (scenario_weights_exponential * np.maximum(reference_eight_portfolio_losses - reference_eight_var_95_loss, 0.0)).sum() / TAIL_PROBABILITY)
reference_eight_weighted_mean_return = float(scenario_weights_exponential @ reference_eight_portfolio_returns)

block4_controls = {
    'requested_symbols_selected_in_requested_order': reference_eight_symbols == REFERENCE_EIGHT_SYMBOLS and len(set(reference_eight_symbols)) == 8,
    'same_dates_and_exponential_probabilities_as_block2': reference_eight_returns.shape == (scenario_count, 8) and len(scenario_dates) == 2204 and np.isclose(scenario_weights_exponential.sum(), 1.0, atol=1e-12, rtol=1e-12),
    'equal_weights_fully_invested_without_shorting': np.all(reference_eight_weights == 0.125) and np.all(reference_eight_weights >= 0.0) and np.isclose(reference_eight_weights.sum(), 1.0, atol=1e-12, rtol=1e-12),
    'paper_amounts_match_budget': np.allclose(reference_eight_target_amounts, 125_000.0, atol=1e-12, rtol=1e-12) and np.isclose(reference_eight_target_amounts.sum(), REFERENCE_BUDGET_USD, atol=1e-8, rtol=1e-12),
    'tail_mass_is_exactly_five_percent': np.isclose(reference_eight_tail_mass.sum(), TAIL_PROBABILITY, atol=1e-12, rtol=1e-12),
    'cvar_is_at_least_var_and_reconciles': reference_eight_cvar_95_loss + 1e-12 >= reference_eight_var_95_loss and np.isclose(reference_eight_cvar_95_loss, reference_eight_cvar_independent, atol=1e-12, rtol=1e-12),
    'full_125_instrument_diagnostic_preserved': portfolio_returns_equal.shape == (scenario_count,) and portfolio_losses_equal.shape == (scenario_count,),
}
if not all(block4_controls.values()):
    raise AssertionError([name for name, passed in block4_controls.items() if not passed])

reference_eight_weights_table = pl.DataFrame({
    'symbol': reference_eight_symbols,
    'portfolio_weight': reference_eight_weights,
    'portfolio_weight_percent': 100.0 * reference_eight_weights,
    'paper_target_amount_usd': reference_eight_target_amounts,
})
reference_eight_risk_summary = pl.DataFrame({
    'metric': ['weighted_mean_return', 'VaR_95_loss', 'CVaR_95_loss'],
    'value_decimal': [reference_eight_weighted_mean_return, reference_eight_var_95_loss, reference_eight_cvar_95_loss],
    'value_percent_of_capital': [100.0 * reference_eight_weighted_mean_return, 100.0 * reference_eight_var_95_loss, 100.0 * reference_eight_cvar_95_loss],
    'horizon': ['one session', 'one session', 'one session'],
})
display(reference_eight_weights_table)
display(reference_eight_risk_summary)
display(pl.DataFrame({'control': list(block4_controls), 'result': ['PASS' if passed else 'FAIL' for passed in block4_controls.values()]}))
print('El resultado de 125 instrumentos permanece como diagnóstico separado; este bloque sólo fija una referencia de ocho ETFs y no optimiza.')


symbol,portfolio_weight,portfolio_weight_percent,paper_target_amount_usd
str,f64,f64,f64
"""SPY""",0.125,12.5,125000.0
"""VEA""",0.125,12.5,125000.0
"""EEM""",0.125,12.5,125000.0
"""TLT""",0.125,12.5,125000.0
"""BIL""",0.125,12.5,125000.0
"""GLD""",0.125,12.5,125000.0
"""USO""",0.125,12.5,125000.0
"""VNQ""",0.125,12.5,125000.0


metric,value_decimal,value_percent_of_capital,horizon
str,f64,f64,str
"""weighted_mean_return""",0.000426,0.04259,"""one session"""
"""VaR_95_loss""",0.010207,1.020723,"""one session"""
"""CVaR_95_loss""",0.016414,1.641393,"""one session"""


control,result
str,str
"""requested_symbols_selected_in_…","""PASS"""
"""same_dates_and_exponential_pro…","""PASS"""
"""equal_weights_fully_invested_w…","""PASS"""
"""paper_amounts_match_budget""","""PASS"""
"""tail_mass_is_exactly_five_perc…","""PASS"""
"""cvar_is_at_least_var_and_recon…","""PASS"""
"""full_125_instrument_diagnostic…","""PASS"""


El resultado de 125 instrumentos permanece como diagnóstico separado; este bloque sólo fija una referencia de ocho ETFs y no optimiza.


## 5. ¿Qué pesos minimizan la pérdida de cola?

Comparamos la referencia de pesos iguales (12.5% por ETF) con una cartera que minimiza CVaR al 95%. Ambas usan exactamente los mismos ocho ETFs, las mismas 2,204 filas históricas y las mismas probabilidades exponenciales del bloque 4. Fijamos antes de optimizar: suma de pesos igual a uno, cada peso entre 0% y 15%, sin posiciones cortas, margen ni efectivo. BIL es un ETF y cuenta dentro del tope. Como ningún peso puede superar 15%, la cartera necesita al menos siete posiciones. Reportamos retorno medio estimado, pero no imponemos un objetivo de retorno.

Para hacer lineal el CVaR introducimos un umbral auxiliar $z$ y excesos $u_s$. Con $L_s(w)=-r_s^\top w$, resolvemos:

$$
\min_{w,z,u}\quad z+\frac{1}{0.05}\sum_s p_s u_s
\quad\text{sujeto a}\quad
u_s\geq L_s(w)-z,\quad u_s\geq0,\quad\sum_i w_i=1,\quad 0\leq w_i\leq0.15.
$$

Aquí $z$ queda libre de signo y representa un candidato a umbral de pérdida; $u_s$ recoge sólo el exceso de cada escenario sobre ese umbral. La mejora se medirá dentro de los escenarios usados para optimizar. Su comportamiento histórico con rebalanceo y costos queda para 07.

In [4]:
from scipy.optimize import linprog

CVAR_CONFIDENCE = 0.95
CVAR_TAIL_PROBABILITY = 1.0 - CVAR_CONFIDENCE
WEIGHT_CAP = 0.15
LP_FEASIBILITY_TOLERANCE = 1e-9
LP_OBJECTIVE_TOLERANCE = 1e-8
optimization_scenarios, optimization_assets = reference_eight_returns.shape
weight_slice = slice(0, optimization_assets)
threshold_index = optimization_assets
excess_slice = slice(optimization_assets + 1, optimization_assets + 1 + optimization_scenarios)

# Central scenario-loss calculation: each row is a joint return scenario and w is the portfolio-weight vector.
reference_eight_losses_for_lp = -(reference_eight_returns @ reference_eight_weights)
objective = np.concatenate((np.zeros(optimization_assets), np.array([1.0]), scenario_weights_exponential / CVAR_TAIL_PROBABILITY))
loss_excess_constraints = np.zeros((optimization_scenarios, optimization_assets + 1 + optimization_scenarios))
loss_excess_constraints[:, weight_slice] = -reference_eight_returns
loss_excess_constraints[:, threshold_index] = -1.0
loss_excess_constraints[:, excess_slice] = -np.eye(optimization_scenarios)
full_investment_constraint = np.zeros((1, optimization_assets + 1 + optimization_scenarios))
full_investment_constraint[0, weight_slice] = 1.0
bounds = [(0.0, WEIGHT_CAP)] * optimization_assets + [(None, None)] + [(0.0, None)] * optimization_scenarios
optimization_result = linprog(
    c=objective,
    A_ub=loss_excess_constraints,
    b_ub=np.zeros(optimization_scenarios),
    A_eq=full_investment_constraint,
    b_eq=np.array([1.0]),
    bounds=bounds,
    method='highs',
)
if not optimization_result.success:
    raise RuntimeError(f'HiGHS failed to solve the CVaR program: status={optimization_result.status}; message={optimization_result.message}')

minimum_cvar_weights = optimization_result.x[weight_slice]
minimum_cvar_threshold_z = float(optimization_result.x[threshold_index])
minimum_cvar_excesses = optimization_result.x[excess_slice]
minimum_cvar_losses = -(reference_eight_returns @ minimum_cvar_weights)

def weighted_tail_metrics(losses, probabilities, tail_probability):
    ascending = np.argsort(losses, kind='stable')
    var_index = int(np.searchsorted(np.cumsum(probabilities[ascending]), 1.0 - tail_probability, side='left'))
    var_loss = float(losses[ascending][var_index])
    descending = np.argsort(-losses, kind='stable')
    sorted_losses = losses[descending]; sorted_probabilities = probabilities[descending]
    probability_before = np.concatenate(([0.0], np.cumsum(sorted_probabilities[:-1])))
    included_probability = np.clip(tail_probability - probability_before, 0.0, sorted_probabilities)
    cvar_loss = float((included_probability * sorted_losses).sum() / tail_probability)
    cvar_identity = float(var_loss + (probabilities * np.maximum(losses - var_loss, 0.0)).sum() / tail_probability)
    return {'var': var_loss, 'cvar': cvar_loss, 'cvar_identity': cvar_identity, 'tail_mass': float(included_probability.sum())}

reference_eight_tail_metrics = weighted_tail_metrics(reference_eight_portfolio_losses, scenario_weights_exponential, CVAR_TAIL_PROBABILITY)
minimum_cvar_tail_metrics = weighted_tail_metrics(minimum_cvar_losses, scenario_weights_exponential, CVAR_TAIL_PROBABILITY)
reference_eight_mean_return = float(scenario_weights_exponential @ reference_eight_portfolio_returns)
minimum_cvar_mean_return = float(scenario_weights_exponential @ (reference_eight_returns @ minimum_cvar_weights))
reference_eight_hhi = float(np.sum(reference_eight_weights**2))
minimum_cvar_hhi = float(np.sum(minimum_cvar_weights**2))

block5_controls = {
    'reference_is_feasible_under_the_same_cap': np.isclose(reference_eight_weights.sum(), 1.0, atol=LP_FEASIBILITY_TOLERANCE, rtol=LP_FEASIBILITY_TOLERANCE) and np.all(reference_eight_weights >= -LP_FEASIBILITY_TOLERANCE) and np.all(reference_eight_weights <= WEIGHT_CAP + LP_FEASIBILITY_TOLERANCE),
    'candidate_weights_feasible_without_post_processing': np.isclose(minimum_cvar_weights.sum(), 1.0, atol=LP_FEASIBILITY_TOLERANCE, rtol=LP_FEASIBILITY_TOLERANCE) and np.all(minimum_cvar_weights >= -LP_FEASIBILITY_TOLERANCE) and np.all(minimum_cvar_weights <= WEIGHT_CAP + LP_FEASIBILITY_TOLERANCE),
    'candidate_requires_at_least_seven_positions': int(np.count_nonzero(minimum_cvar_weights > LP_FEASIBILITY_TOLERANCE)) >= 7,
    'cvar_tail_mass_and_independent_identity': all(np.isclose(metrics['tail_mass'], CVAR_TAIL_PROBABILITY, atol=1e-12, rtol=1e-12) and np.isclose(metrics['cvar'], metrics['cvar_identity'], atol=1e-12, rtol=1e-12) for metrics in [reference_eight_tail_metrics, minimum_cvar_tail_metrics]),
    'solver_objective_reconciles_with_candidate_cvar': np.isclose(float(optimization_result.fun), minimum_cvar_tail_metrics['cvar'], atol=LP_OBJECTIVE_TOLERANCE, rtol=LP_OBJECTIVE_TOLERANCE),
    'candidate_cvar_no_higher_than_feasible_reference': minimum_cvar_tail_metrics['cvar'] <= reference_eight_tail_metrics['cvar'] + LP_OBJECTIVE_TOLERANCE,
    'loss_excess_constraints_satisfied': np.all(minimum_cvar_excesses >= minimum_cvar_losses - minimum_cvar_threshold_z - LP_FEASIBILITY_TOLERANCE) and np.all(minimum_cvar_excesses >= -LP_FEASIBILITY_TOLERANCE),
}
if not all(block5_controls.values()):
    raise AssertionError([name for name, passed in block5_controls.items() if not passed])

minimum_cvar_weight_table = pl.DataFrame({
    'symbol': REFERENCE_EIGHT_SYMBOLS,
    'reference_equal_weight': reference_eight_weights,
    'minimum_cvar_weight': minimum_cvar_weights,
    'weight_difference_percentage_points': 100.0 * (minimum_cvar_weights - reference_eight_weights),
    'candidate_target_amount_usd': REFERENCE_BUDGET_USD * minimum_cvar_weights,
    'at_15_percent_cap': np.isclose(minimum_cvar_weights, WEIGHT_CAP, atol=LP_FEASIBILITY_TOLERANCE, rtol=LP_FEASIBILITY_TOLERANCE),
})
comparison_summary = pl.DataFrame({
    'portfolio': ['equal_weight_reference', 'minimum_CVaR_95'],
    'weighted_mean_return_percent': [100.0 * reference_eight_mean_return, 100.0 * minimum_cvar_mean_return],
    'VaR_95_loss_percent': [100.0 * reference_eight_tail_metrics['var'], 100.0 * minimum_cvar_tail_metrics['var']],
    'CVaR_95_loss_percent': [100.0 * reference_eight_tail_metrics['cvar'], 100.0 * minimum_cvar_tail_metrics['cvar']],
    'weight_HHI': [reference_eight_hhi, minimum_cvar_hhi],
    'effective_number_of_holdings': [1.0 / reference_eight_hhi, 1.0 / minimum_cvar_hhi],
})
cvar_reduction_percentage_points = 100.0 * (reference_eight_tail_metrics['cvar'] - minimum_cvar_tail_metrics['cvar'])
cvar_reduction_usd = REFERENCE_BUDGET_USD * (reference_eight_tail_metrics['cvar'] - minimum_cvar_tail_metrics['cvar'])
display(minimum_cvar_weight_table)
display(comparison_summary)
display(pl.DataFrame({'control': list(block5_controls), 'result': ['PASS' if passed else 'FAIL' for passed in block5_controls.values()]}))
print(f'HiGHS status: {optimization_result.message}; z={minimum_cvar_threshold_z:.8f}; CVaR reduction versus equal weights={cvar_reduction_percentage_points:.4f} percentage points, or USD {cvar_reduction_usd:,.2f} on the paper budget.')
print('The reduction is measured on the same weighted scenarios used for optimization; it is not evidence of realized performance after rebalancing or costs.')


symbol,reference_equal_weight,minimum_cvar_weight,weight_difference_percentage_points,candidate_target_amount_usd,at_15_percent_cap
str,f64,f64,f64,f64,bool
"""SPY""",0.125,0.15,2.5,150000.0,true
"""VEA""",0.125,0.15,2.5,150000.0,true
"""EEM""",0.125,0.019206,-10.579423,19205.770945,false
"""TLT""",0.125,0.15,2.5,150000.0,true
"""BIL""",0.125,0.15,2.5,150000.0,true
"""GLD""",0.125,0.15,2.5,150000.0,true
"""USO""",0.125,0.083482,-4.151845,83481.551904,false
"""VNQ""",0.125,0.147313,2.231268,147312.677151,false


portfolio,weighted_mean_return_percent,VaR_95_loss_percent,CVaR_95_loss_percent,weight_HHI,effective_number_of_holdings
str,f64,f64,f64,f64,f64
"""equal_weight_reference""",0.04259,1.020723,1.641393,0.125,8.0
"""minimum_CVaR_95""",0.039129,0.961787,1.485424,0.141539,7.065188


control,result
str,str
"""reference_is_feasible_under_th…","""PASS"""
"""candidate_weights_feasible_wit…","""PASS"""
"""candidate_requires_at_least_se…","""PASS"""
"""cvar_tail_mass_and_independent…","""PASS"""
"""solver_objective_reconciles_wi…","""PASS"""
"""candidate_cvar_no_higher_than_…","""PASS"""
"""loss_excess_constraints_satisf…","""PASS"""


HiGHS status: Optimization terminated successfully. (HiGHS Status 7: Optimal); z=0.00961787; CVaR reduction versus equal weights=0.1560 percentage points, or USD 1,559.69 on the paper budget.
The reduction is measured on the same weighted scenarios used for optimization; it is not evidence of realized performance after rebalancing or costs.


## 6. ¿Qué conseguimos al cambiar los pesos?

Frente al 12.5% por ETF, la cartera candidata reduce el CVaR al 95% de 1.6414% a 1.4854% por sesión: una diferencia de 0.1560 puntos porcentuales, aproximadamente USD 1,560 sobre el presupuesto *paper* de USD 1 millón. Es una diferencia estimada de pérdida media de cola, no un ahorro realizado.

El intercambio observado es un retorno medio ponderado de 0.0426% a 0.0391% y un número efectivo de posiciones de 8.00 a 7.07. SPY, VEA, TLT, BIL y GLD alcanzan el tope de 15%; EEM y USO reducen su peso. Estos cambios se describen como resultado conjunto de la optimización y no atribuyen por separado la mejora del CVaR a un ETF.

La notebook 06 entrega pesos y montos objetivo para una fecha de corte. La reducción de CVaR se obtiene en los escenarios usados para optimizar. En 07 evaluaremos la regla de asignación fuera de su muestra de ajuste en cada fecha, con rebalanceo y costos, frente a la misma referencia de pesos iguales.